<a href="https://colab.research.google.com/github/mcfvdesigns/Site-Turnover-Defects/blob/main/Baseline_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q ultralytics pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 916.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 4.2 MB/s eta 0:00:00


In [2]:
import os, hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/mcfvdesigns/Site-Turnover-Defects/releases/download/v4.0/Site.Turnover.Defects.v4i.yolov11.1.zip"
SHA256 = "443eced074cc61c29b5fd3e0b8a0ce3bb5d6de410312312acc5ce936a10a84bc"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]

already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)

{'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 5, 'names': ['cracked jamb', 'cracked tile', 'moisture stain', 'paint defect', 'uneven joint'], 'roboflow': {'workspace': 'skipper-_sky', 'project': 'site-turnover-defects', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/skipper-_sky/site-turnover-defects/dataset/4'}, 'path': '/content/dataset'}


In [3]:
from ultralytics import YOLO

# 1. Load an un-finetuned, pre-trained base model
baseline_model = YOLO("yolo11n.pt")

# 2. Run baseline inference on test images
results = baseline_model.predict(
    source=str(dataset_root / "test/images"),
    save=True,
    conf=0.25
)

# 3. Evaluate baseline zero-shot performance on the validation split
metrics = baseline_model.val(data=str(cfg_path))
print("Baseline mAP50-95:", metrics.box.map)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

image 1/5 /content/dataset/test/images/39C7A391-E9C7-4E96-9517-6D375401DEB8_heic.rf.5db3a079d7a90bf32e4eb67064162c4b.jpg: 640x640 (no detections), 403.1ms
image 2/5 /content/dataset/test/images/3B0369CD-D1EE-472C-9F32-4B2AF3395297_heic.rf.a5cc760ef037a0c61012c6bef46a2a9a.jpg: 640x640 (no detections), 179.4ms
image 3/5 /content/dataset/test/images/3C0BF6B7-BD58-4F89-9809-2541F0FF267A_heic.rf.b349f558e2e2c53882e350e3996622b9.jpg: 640x640 (no detections), 183.0ms
image 4/5 /content/dataset/test/images/5D20FB4A-8995-4278-8FB0-38F0F16F3222_heic.rf.90e31dfec36c4df4657b053fab35876d.jpg: 640x640 1 motorcycle, 1 bed, 171.1ms
image 5/5 /content/dataset/test/images/6CCD60CE-BE5E-41A1-8675-DFD83020914E_heic.r